In [6]:
import pandas as pd
import plotly.graph_objects as go

file_path = r"C:\Users\satya\Downloads\Play Store Data.csv"
df = pd.read_csv(file_path, dtype={'Price': str})  # Ensure 'Price' is treated as a string

df['Installs'] = df['Installs'].str.replace(r'[,+]', '', regex=True)
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')

df['Price'] = df['Price'].astype(str).str.replace(r'[₹$,]', '', regex=True)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

df['Size'] = df['Size'].replace('Varies with device', pd.NA)
df['Size'] = df['Size'].str.replace(r'[M]', '', regex=True).replace('', pd.NA)
df['Size'] = pd.to_numeric(df['Size'], errors='coerce')

df['Android Ver'] = df['Android Ver'].str.extract(r'(\d+\.\d+)').astype(float)

df = df.dropna(subset=['Installs', 'Price', 'Size', 'Android Ver'])

df_filtered = df[
    (df['Installs'] >= 10000) &  # Minimum installs: 10,000
    (df['Size'] > 15) &  # Minimum app size: 15M
    (df['Content Rating'] == 'Everyone') &  # Only apps rated "Everyone"
    (df['App'].str.len() <= 30) &  # App name max length: 30 chars
    (df['Android Ver'] > 4.0)  # Minimum Android version: 4.0+
]

df_filtered['Revenue'] = df_filtered['Installs'] * df_filtered['Price']
df_filtered['Revenue'].fillna(0, inplace=True)

top_categories = df_filtered['Category'].value_counts().nlargest(3).index
df_top_categories = df_filtered[df_filtered['Category'].isin(top_categories)]

df_free_vs_paid = df_top_categories.groupby(['Category', 'Type']).agg(
    avg_installs=('Installs', 'mean'),
    avg_revenue=('Revenue', 'mean')
).reset_index()

df_free = df_free_vs_paid[df_free_vs_paid['Type'] == 'Free']
df_paid = df_free_vs_paid[df_free_vs_paid['Type'] == 'Paid']

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    x=df_free['Category'],
    y=df_free['avg_installs'],
    name='Free Apps - Avg Installs',
    marker=dict(color='#0b08bc'),
    yaxis='y1'
))

fig2.add_trace(go.Bar(
    x=df_paid['Category'],
    y=df_paid['avg_installs'],
    name='Paid Apps - Avg Installs',
    marker=dict(color='#ff4500'),
    yaxis='y1'
))

fig2.add_trace(go.Scatter(
    x=df_free['Category'],
    y=df_free['avg_revenue'],
    mode='lines+markers',
    name='Free Apps - Avg Revenue',
    marker=dict(color='#3b7a57'),
    yaxis='y2'
))

fig2.add_trace(go.Scatter(
    x=df_paid['Category'],
    y=df_paid['avg_revenue'],
    mode='lines+markers',
    name='Paid Apps - Avg Revenue',
    marker=dict(color='#800020'),
    yaxis='y2'
))

fig2.update_layout(
    margin=dict(l=0, r=0, t=30, b=30),  # Increased right margin
    title='Comparison of Average Installs and Revenue for Free vs Paid Apps',
    xaxis=dict(title='Category', tickangle=-30),
    yaxis=dict(
        title='Average Installs',
        titlefont=dict(color='#0b08bc'),
        tickfont=dict(color='#0b08bc'),
        type='log',  # Log scale for installs
        tickvals=[10_000, 100_000, 1_000_000, 10_000_000, 100_000_000],
        ticktext=["10K", "100K", "1M", "10M", "100M"]
    ),
    yaxis2=dict(
        title='Average Revenue ($)',
        titlefont=dict(color='#800020'),
        tickfont=dict(color='#800020'),
        overlaying='y',
        side='right',
        tickvals=[100_000, 2_000_000, 5_000_000],
        ticktext=["100K", "200K", "500K",]
    ),
    barmode='group',
    template="plotly_white"
)

fig2.show()
html_file_path = r"C:\Users\satya\OneDrive\Desktop\Satyam\Task 2.html"
fig2.write_html(html_file_path)  

print(f" Plot saved successfully at: {html_file_path}", "Free apps in all three categories (FAMILY, GAME, SPORTS) have much higher average installs compared to paid apps.")


C:\Users\satya\AppData\Local\Temp\ipykernel_384\4008088655.py:29: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\satya\AppData\Local\Temp\ipykernel_384\4008088655.py:30: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.



C:\Users\satya\AppData\Local\Temp\ipykernel_384\4008088655.py:30

 Plot saved successfully at: C:\Users\satya\OneDrive\Desktop\Satyam\Task 2.html Free apps in all three categories (FAMILY, GAME, SPORTS) have much higher average installs compared to paid apps.
